# Byte Sequence Autoencoder

Train a simple autoencoder that compresses byte sequences to a small latent vector
and reconstructs them. Loss is cross-entropy per byte position.

Supports three granularity modes:
- **word**: extract letter-only words (e.g. `hello`), pad to `SEQ_LEN`
- **sentence**: extract spans between sentence boundaries (`.!?` followed by space/newline), pad to `SEQ_LEN`
- **stream**: fixed-length chunks of the raw byte stream (no padding needed)

Configure the mode in the setup cell below.

In [ ]:
import glob
import numpy as np
from pathlib import Path

PLOT_DIR = Path("plots/explore_word_autoencoder")
PLOT_DIR.mkdir(parents=True, exist_ok=True)
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from efficient_byte_tokenizer import EfficientByteTokenizer, ByteCategory

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

## Load data and extract sequences

In [ ]:
# --- Configuration ---
DATA_PATH = "data/datasets/fineweb10B_byte260"
train_pattern = f"{DATA_PATH}/fineweb_train_*.bin"
val_pattern = f"{DATA_PATH}/fineweb_val_*.bin"
MAX_SHARDS = 11
MAX_TOKENS = 500_000_000
MAX_VAL_TOKENS = 10_000_000

_eff_tok = EfficientByteTokenizer()
V = _eff_tok.vocab_size
print(_eff_tok.describe())


def load_data_shard(file: Path) -> np.ndarray:
    header_bytes = 256 * np.dtype("<i4").itemsize
    header = np.fromfile(file, dtype="<i4", count=256)
    assert header[0] == 20240520 and header[1] == 1, f"Bad header in {file}"
    num_tokens = int(header[2])
    tokens = np.fromfile(file, dtype="<u2", count=num_tokens, offset=header_bytes)
    return tokens


def load_and_remap(pattern, max_shards=None, max_tokens=0):
    files = sorted(glob.glob(pattern))
    if max_shards:
        files = files[:max_shards]
    print(f"Loading {len(files)} shards...")
    tokens = np.concatenate([load_data_shard(Path(f)) for f in files])
    tokens = _eff_tok.filter_stream(_eff_tok.remap_byte260_shard(tokens))
    if max_tokens > 0 and len(tokens) > max_tokens:
        tokens = tokens[:max_tokens]
    print(f"  {len(tokens):,} tokens (V={V})")
    return tokens


train_tokens = load_and_remap(train_pattern, MAX_SHARDS, MAX_TOKENS)
val_tokens = load_and_remap(val_pattern, max_tokens=MAX_VAL_TOKENS)

In [ ]:
# ================================================================
# Granularity mode — change this to switch between word/sentence/stream
# ================================================================
MODE = "stream"  # one of: "word", "sentence", "stream"

# Per-mode settings
if MODE == "word":
    SEQ_LEN = 30  # max word length in bytes
    MIN_SEQ_LEN = 2  # min word length
elif MODE == "sentence":
    SEQ_LEN = 128  # max sentence length in bytes
    MIN_SEQ_LEN = 5  # min sentence length
elif MODE == "stream":
    SEQ_LEN = 128  # fixed chunk length
    MIN_SEQ_LEN = SEQ_LEN  # all chunks are exactly SEQ_LEN
else:
    raise ValueError(f"Unknown MODE: {MODE!r}")

MAX_WORD_LEN = SEQ_LEN  # alias used by model definitions

print(f"Mode: {MODE}  |  SEQ_LEN={SEQ_LEN}  MIN_SEQ_LEN={MIN_SEQ_LEN}")

# --- Shared: token-to-byte lookup ---
tok_to_byte = np.full(V, 0, dtype=np.uint8)
for b in range(256):
    tid = int(_eff_tok._byte_to_id[b])
    if tid < V:
        tok_to_byte[tid] = b

# --- Shared: letter mask (needed for word mode, also useful for sentence) ---
_letter_ids = set()
for cat in [ByteCategory.UPPERCASE, ByteCategory.LOWERCASE]:
    _letter_ids.update(_eff_tok.ids(cat).tolist())
is_letter_tok = np.zeros(V, dtype=bool)
for t in _letter_ids:
    is_letter_tok[t] = True


def _word_start_mask(tokens):
    letter_mask = is_letter_tok[tokens]
    starts = np.zeros(len(tokens), dtype=bool)
    starts[0] = letter_mask[0]
    starts[1:] = letter_mask[1:] & ~letter_mask[:-1]
    return starts


# ================================================================
# Extraction functions
# ================================================================


def extract_words(tokens):
    """Extract letter-only words as zero-padded byte arrays."""
    letter_mask = is_letter_tok[tokens]
    ws = _word_start_mask(tokens)
    we = np.zeros(len(tokens), dtype=bool)
    we[:-1] = letter_mask[:-1] & ~letter_mask[1:]
    we[-1] = letter_mask[-1]

    ws_pos = np.where(ws)[0]
    we_pos = np.where(we)[0]
    assert len(ws_pos) == len(we_pos)

    span_lens = we_pos - ws_pos + 1
    valid = (span_lens >= MIN_SEQ_LEN) & (span_lens <= SEQ_LEN)
    ws_pos = ws_pos[valid]
    span_lens = span_lens[valid]
    print(f"  Found {len(ws_pos):,} words (len {MIN_SEQ_LEN}-{SEQ_LEN})")

    byte_tokens = tok_to_byte[tokens]
    col_idx = np.arange(SEQ_LEN)[None, :]
    src_idx = np.clip(ws_pos[:, None] + col_idx, 0, len(byte_tokens) - 1)
    in_range = col_idx < span_lens[:, None]
    return byte_tokens[src_idx] * in_range.astype(np.uint8)


def extract_sentences(tokens):
    """Extract sentences (spans between sentence-ending punctuation)."""
    byte_tokens = tok_to_byte[tokens]

    # Sentence boundaries: .!? followed by space, newline, or end of stream
    is_sent_end = np.zeros(len(tokens), dtype=bool)
    for ch in [ord("."), ord("!"), ord("?")]:
        is_sent_end |= byte_tokens == ch

    # Require next token to be space/newline/bos or end of stream
    is_space = np.zeros(len(tokens), dtype=bool)
    for ch in [ord(" "), ord("\n"), ord("\r"), ord("\t")]:
        is_space |= byte_tokens == ch

    # Sentence end = punctuation followed by space-like or end
    sent_end = np.zeros(len(tokens), dtype=bool)
    sent_end[:-1] = is_sent_end[:-1] & is_space[1:]
    sent_end[-1] = is_sent_end[-1]

    end_pos = np.where(sent_end)[0]
    if len(end_pos) == 0:
        return np.zeros((0, SEQ_LEN), dtype=np.uint8)

    # Sentence spans: start after previous end+1 (skip the space)
    start_pos = np.empty_like(end_pos)
    start_pos[0] = 0
    start_pos[1:] = end_pos[:-1] + 2  # skip punctuation + space
    # Include the punctuation in the sentence
    end_pos_incl = end_pos + 1

    span_lens = end_pos_incl - start_pos
    valid = (span_lens >= MIN_SEQ_LEN) & (span_lens <= SEQ_LEN) & (start_pos >= 0)
    start_pos = start_pos[valid]
    span_lens = span_lens[valid]
    print(f"  Found {len(start_pos):,} sentences (len {MIN_SEQ_LEN}-{SEQ_LEN})")

    col_idx = np.arange(SEQ_LEN)[None, :]
    src_idx = np.clip(start_pos[:, None] + col_idx, 0, len(byte_tokens) - 1)
    in_range = col_idx < span_lens[:, None]
    return byte_tokens[src_idx] * in_range.astype(np.uint8)


def extract_stream_chunks(tokens):
    """Extract fixed-length non-overlapping chunks of the byte stream."""
    byte_tokens = tok_to_byte[tokens]
    n_chunks = len(byte_tokens) // SEQ_LEN
    print(f"  Extracting {n_chunks:,} chunks of length {SEQ_LEN}")
    trimmed = byte_tokens[: n_chunks * SEQ_LEN]
    return trimmed.reshape(n_chunks, SEQ_LEN)


# ================================================================
# Run extraction
# ================================================================
_extract_fn = {
    "word": extract_words,
    "sentence": extract_sentences,
    "stream": extract_stream_chunks,
}[MODE]

print(f"Extracting train sequences ({MODE})...")
train_seqs_all = _extract_fn(train_tokens)

print(f"Extracting val sequences ({MODE})...")
val_words = _extract_fn(val_tokens)

# Deduplicate train sequences
print(f"\nDeduplicating train sequences...")
train_view = train_seqs_all.view(np.dtype((np.void, SEQ_LEN)))
uniq_bytes, inverse, counts = np.unique(
    train_view, return_inverse=True, return_counts=True
)
train_words = uniq_bytes.view(np.uint8).reshape(-1, SEQ_LEN)
train_word_freq = counts.astype(np.float32)
train_word_weights = train_word_freq / train_word_freq.sum()

print(
    f"Unique train sequences: {len(train_words):,} "
    f"(from {len(train_seqs_all):,} occurrences, "
    f"{len(train_words) / len(train_seqs_all):.1%} unique)"
)
del train_seqs_all

# Length distribution (non-pad bytes per sequence)
lengths = (train_words > 0).sum(axis=1)
print(
    f"Length stats: mean={lengths.mean():.1f}, "
    f"median={np.median(lengths):.0f}, max={lengths.max()}"
)

fig, ax = plt.subplots(figsize=(10, 4))
if MODE == "stream":
    ax.hist(lengths, bins=50, edgecolor="black", alpha=0.7, weights=train_word_freq)
else:
    ax.hist(
        lengths,
        bins=np.arange(MIN_SEQ_LEN, SEQ_LEN + 2) - 0.5,
        edgecolor="black",
        alpha=0.7,
        weights=train_word_freq,
    )
ax.set_xlabel("Sequence length (bytes)")
ax.set_ylabel("Frequency")
ax.set_title(
    f"{MODE.title()} length distribution ({len(train_words):,} unique, "
    f"{int(train_word_freq.sum()):,} total)"
)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Model definitions

In [ ]:
NUM_BYTES = 256  # byte values 1-255 are valid, 0 = padding
NUM_CLASSES = NUM_BYTES + 1  # include padding class


class MLPAutoencoder(nn.Module):
    """Simple MLP autoencoder with flatten/unflatten."""

    def __init__(
        self, max_len=30, embed_dim=32, hidden_dim=256, latent_dim=32, n_hidden=2
    ):
        super().__init__()
        self.max_len = max_len
        self.embed = nn.Embedding(NUM_CLASSES, embed_dim, padding_idx=0)

        enc_layers = [nn.Linear(max_len * embed_dim, hidden_dim), nn.GELU()]
        for _ in range(n_hidden - 1):
            enc_layers += [nn.Linear(hidden_dim, hidden_dim), nn.GELU()]
        enc_layers.append(nn.Linear(hidden_dim, latent_dim))
        self.encoder = nn.Sequential(*enc_layers)

        dec_layers = [nn.Linear(latent_dim, hidden_dim), nn.GELU()]
        for _ in range(n_hidden - 1):
            dec_layers += [nn.Linear(hidden_dim, hidden_dim), nn.GELU()]
        dec_layers.append(nn.Linear(hidden_dim, max_len * NUM_CLASSES))
        self.decoder = nn.Sequential(*dec_layers)

    def encode(self, x):
        emb = self.embed(x).reshape(x.shape[0], -1)
        return self.encoder(emb)

    def decode(self, z):
        return self.decoder(z).reshape(z.shape[0], self.max_len, NUM_CLASSES)

    def forward(self, x):
        z = self.encode(x)
        return self.decode(z), z


class ConvAutoencoder(nn.Module):
    """1D convolutional autoencoder with configurable depth.

    Args:
        n_downsample: number of stride-2 conv layers (spatial reduction = 2^n_downsample).
        n_conv: total conv layers in encoder. Extra layers beyond n_downsample use stride=1.
        pool: "stride" (strided conv, default), "avg" (avg pool + conv), "max" (max pool + conv).
        tied_weights: if True, decoder reuses encoder conv weights via F.conv_transpose1d.
            Only compatible with pool="stride". Halves the conv parameter count.
    """

    def __init__(
        self,
        max_len=30,
        embed_dim=32,
        channels=128,
        latent_dim=32,
        n_downsample=2,
        n_conv=3,
        pool="stride",
        tied_weights=False,
    ):
        super().__init__()
        if tied_weights and pool != "stride":
            raise ValueError(
                f"tied_weights requires pool='stride', got pool={pool!r}. "
                f"Pooling layers have no weights to tie — only strided convs "
                f"have an exact transpose."
            )

        self.max_len = max_len
        self.channels = channels
        self.tied_weights = tied_weights
        self.n_downsample = n_downsample
        self.n_conv = n_conv
        self.pool = pool
        self.embed = nn.Embedding(NUM_CLASSES, embed_dim, padding_idx=0)

        # --- Determine which layers downsample ---
        # Spread downsample layers evenly: always start with stride-1 (layer 0),
        # then interleave stride-2 among remaining layers.
        # E.g. n_conv=5, n_downsample=2 -> strides [1, 1, 2, 1, 2]
        #      n_conv=4, n_downsample=2 -> strides [1, 2, 1, 2]
        #      n_conv=3, n_downsample=2 -> strides [1, 2, 2]
        self._layer_downsamples = [False] * n_conv
        if n_downsample > 0:
            avail = list(range(1, n_conv))
            for di in range(n_downsample):
                idx = avail[round(di * (len(avail) - 1) / max(n_downsample - 1, 1))]
                self._layer_downsamples[idx] = True

        # --- Encoder conv layers (stored individually for tying) ---
        self.enc_convs = nn.ModuleList()
        self.enc_strides = []
        self.enc_paddings = []
        in_ch = embed_dim
        for i in range(n_conv):
            do_down = self._layer_downsamples[i]
            if do_down:
                if pool == "stride":
                    self.enc_convs.append(
                        nn.Conv1d(in_ch, channels, 3, stride=2, padding=1)
                    )
                    self.enc_strides.append(2)
                    self.enc_paddings.append(1)
                elif pool in ("avg", "max"):
                    self.enc_convs.append(nn.Conv1d(in_ch, channels, 3, padding=1))
                    self.enc_strides.append(1)
                    self.enc_paddings.append(1)
                else:
                    raise ValueError(f"Unknown pool: {pool!r}")
            else:
                self.enc_convs.append(nn.Conv1d(in_ch, channels, 3, padding=1))
                self.enc_strides.append(1)
                self.enc_paddings.append(1)
            in_ch = channels

        # Pool layers for non-stride modes
        self.enc_pools = nn.ModuleList()
        self.enc_has_pool = []
        for i in range(n_conv):
            do_down = self._layer_downsamples[i]
            if do_down and pool == "avg":
                self.enc_pools.append(nn.AvgPool1d(2))
                self.enc_has_pool.append(True)
            elif do_down and pool == "max":
                self.enc_pools.append(nn.MaxPool1d(2))
                self.enc_has_pool.append(True)
            else:
                self.enc_pools.append(nn.Identity())
                self.enc_has_pool.append(False)

        self.act = nn.GELU()

        # Encoder output length
        self._enc_len = max_len
        for _ in range(n_downsample):
            self._enc_len = (self._enc_len + 1) // 2
        self.enc_fc = nn.Linear(channels * self._enc_len, latent_dim)

        # --- Decoder ---
        self.dec_fc = nn.Linear(latent_dim, channels * self._enc_len)

        if not tied_weights:
            # Independent decoder conv layers
            self.dec_convs = nn.ModuleList()
            for i in range(n_conv):
                # Decoder mirrors encoder in reverse
                enc_i = n_conv - 1 - i
                do_up = self._layer_downsamples[enc_i]
                if do_up:
                    self.dec_convs.append(
                        nn.ConvTranspose1d(
                            channels, channels, 3, stride=2, padding=1, output_padding=1
                        )
                    )
                else:
                    out_ch = channels if enc_i > 0 else channels
                    self.dec_convs.append(nn.Conv1d(channels, out_ch, 3, padding=1))

        # When tied, the first encoder layer (embed_dim -> channels) can't be
        # tied because it changes dimensionality. Add an untied conv instead.
        if tied_weights:
            self.dec_final_conv = nn.Conv1d(channels, channels, 3, padding=1)

        # Output projection (never tied)
        self.dec_out = nn.Conv1d(channels, NUM_CLASSES, 1)

        # Track intermediate sizes for tied decode (to get output_padding right)
        self._enc_sizes = []

    def encode(self, x):
        h = self.embed(x).transpose(1, 2)  # (B, E, L)
        self._enc_sizes = [h.shape[2]]
        for i in range(self.n_conv):
            h = self.enc_convs[i](h)
            h = self.enc_pools[i](h)
            h = self.act(h)
            self._enc_sizes.append(h.shape[2])
        return self.enc_fc(h.reshape(x.shape[0], -1))

    def _tied_decode_convs(self, h):
        """Decode using encoder weights transposed, in reverse order.

        Layers 1..n_conv-1 are tied (conv_transpose1d with encoder weights).
        Layer 0 (embed_dim -> channels) is NOT tied because it changes
        dimensionality; we use self.dec_final_conv instead.
        """
        # Tied layers: encoder layers n_conv-1 down to 1 (skip layer 0)
        for i in range(self.n_conv - 1):
            enc_i = self.n_conv - 1 - i  # mirror index: n_conv-1, n_conv-2, ..., 1
            w = self.enc_convs[enc_i].weight
            b = self.enc_convs[enc_i].bias
            stride = self.enc_strides[enc_i]
            padding = self.enc_paddings[enc_i]

            if stride > 1:
                target_len = self._enc_sizes[enc_i]
                h = F.conv_transpose1d(
                    h,
                    w,
                    bias=b,
                    stride=stride,
                    padding=padding,
                    output_padding=stride - 1,
                )
                if h.shape[2] > target_len:
                    h = h[:, :, :target_len]
                elif h.shape[2] < target_len:
                    h = F.pad(h, (0, target_len - h.shape[2]))
            else:
                h = F.conv_transpose1d(h, w, bias=b, stride=1, padding=padding)
            h = self.act(h)

        # Untied final layer (replaces encoder layer 0 which is embed_dim -> channels)
        h = self.dec_final_conv(h)
        h = self.act(h)
        return h

    def decode(self, z):
        h = self.dec_fc(z).reshape(z.shape[0], self.channels, self._enc_len)
        if self.tied_weights:
            h = self._tied_decode_convs(h)
        else:
            for i in range(self.n_conv):
                h = self.dec_convs[i](h)
                h = self.act(h)
        h = self.dec_out(h)  # (B, NUM_CLASSES, L')
        h = h[:, :, : self.max_len]
        if h.shape[2] < self.max_len:
            h = F.pad(h, (0, self.max_len - h.shape[2]))
        return h.transpose(1, 2)  # (B, L, NUM_CLASSES)

    def forward(self, x):
        z = self.encode(x)
        return self.decode(z), z

    def __repr__(self):
        tied_str = ", tied" if self.tied_weights else ""
        return (
            f"ConvAutoencoder(n_conv={self.n_conv}, n_down={self.n_downsample}, "
            f"pool={self.pool!r}{tied_str}, ch={self.channels}, "
            f"enc_len={self._enc_len}, latent={self.enc_fc.out_features})"
        )


class GRUAutoencoder(nn.Module):
    """GRU encoder, autoregressive GRU decoder."""

    def __init__(
        self, max_len=30, embed_dim=32, hidden_dim=128, latent_dim=32, n_layers=1
    ):
        super().__init__()
        self.max_len = max_len
        self.embed = nn.Embedding(NUM_CLASSES, embed_dim, padding_idx=0)

        self.enc_gru = nn.GRU(
            embed_dim, hidden_dim, num_layers=n_layers, batch_first=True
        )
        self.enc_fc = nn.Linear(hidden_dim, latent_dim)

        self.dec_fc = nn.Linear(latent_dim, hidden_dim)
        self.dec_gru = nn.GRU(
            embed_dim, hidden_dim, num_layers=n_layers, batch_first=True
        )
        self.dec_out = nn.Linear(hidden_dim, NUM_CLASSES)
        self.hidden_dim = hidden_dim
        self.n_layers = n_layers

    def encode(self, x):
        emb = self.embed(x)
        _, h = self.enc_gru(emb)  # h: (n_layers, B, H)
        return self.enc_fc(h[-1])  # use last layer's hidden state

    def decode(self, z, x=None):
        h0 = self.dec_fc(z).unsqueeze(0)  # (1, B, H)
        if self.n_layers > 1:
            h0 = h0.expand(self.n_layers, -1, -1).contiguous()
        B = z.shape[0]
        if x is not None:
            dec_input = torch.zeros(B, 1, dtype=x.dtype, device=x.device)
            dec_input = torch.cat([dec_input, x[:, :-1]], dim=1)
            emb = self.embed(dec_input)
            out, _ = self.dec_gru(emb, h0)
            return self.dec_out(out)
        else:
            outputs = []
            inp = torch.zeros(B, 1, dtype=torch.long, device=z.device)
            h = h0
            for _ in range(self.max_len):
                emb = self.embed(inp)
                out, h = self.dec_gru(emb, h)
                logits = self.dec_out(out)
                outputs.append(logits)
                inp = logits.argmax(dim=-1)
            return torch.cat(outputs, dim=1)

    def forward(self, x):
        z = self.encode(x)
        return self.decode(z, x), z


def count_params(model):
    return sum(p.numel() for p in model.parameters())


# Quick sanity check
test_configs = [
    ("MLP(2 hidden)", MLPAutoencoder(max_len=MAX_WORD_LEN, latent_dim=32)),
    (
        "Conv(3c,2d)",
        ConvAutoencoder(max_len=MAX_WORD_LEN, latent_dim=32, n_conv=3, n_downsample=2),
    ),
    (
        "Conv(5c,3d)",
        ConvAutoencoder(max_len=MAX_WORD_LEN, latent_dim=32, n_conv=5, n_downsample=3),
    ),
    (
        "Conv(3c,avg)",
        ConvAutoencoder(
            max_len=MAX_WORD_LEN,
            latent_dim=32,
            n_conv=3,
            pool="avg",
            tied_weights=False,
        ),
    ),
    (
        "Conv(3c,2d,tied)",
        ConvAutoencoder(
            max_len=MAX_WORD_LEN,
            latent_dim=32,
            n_conv=3,
            n_downsample=2,
            tied_weights=True,
        ),
    ),
    (
        "Conv(5c,3d,tied)",
        ConvAutoencoder(
            max_len=MAX_WORD_LEN,
            latent_dim=32,
            n_conv=5,
            n_downsample=3,
            tied_weights=True,
        ),
    ),
    ("GRU(1 layer)", GRUAutoencoder(max_len=MAX_WORD_LEN, latent_dim=32, n_layers=1)),
    ("GRU(2 layers)", GRUAutoencoder(max_len=MAX_WORD_LEN, latent_dim=32, n_layers=2)),
]

# Verify tied_weights + pool raises error
try:
    ConvAutoencoder(max_len=MAX_WORD_LEN, latent_dim=32, pool="avg", tied_weights=True)
    assert False, "Should have raised ValueError"
except ValueError as e:
    print(f"Correctly rejected tied + pool: {e}")

print(f"{'Config':<25s} {'Params':>10s}  Shape check")
print("-" * 55)
for name, m in test_configs:
    x = torch.randint(0, 256, (4, MAX_WORD_LEN))
    logits, z = m(x)
    ok = logits.shape == (4, MAX_WORD_LEN, NUM_CLASSES) and z.shape == (4, 32)
    print(f"{name:<25s} {count_params(m):>10,}  {'OK' if ok else 'FAIL'}")

## Training

In [ ]:
import time
from torch.utils.data import WeightedRandomSampler


def train_autoencoder(
    model,
    train_data,
    val_data,
    n_epochs=10,
    batch_size=4096,
    lr=1e-3,
    label="",
    train_weights=None,
    samples_per_epoch=500_000,
):
    """Train an autoencoder with frequency-weighted sampling."""
    model = model.to(device)
    train_t = torch.from_numpy(train_data.astype(np.int64))
    val_t = torch.from_numpy(val_data.astype(np.int64)).to(device)

    # Frequency-weighted sampling: oversample common words
    if train_weights is not None:
        sampler = WeightedRandomSampler(
            weights=torch.from_numpy(train_weights),
            num_samples=samples_per_epoch,
            replacement=True,
        )
        train_dl = DataLoader(
            TensorDataset(train_t),
            batch_size=batch_size,
            sampler=sampler,
            num_workers=0,
            pin_memory=(device != "cpu"),
        )
    else:
        train_dl = DataLoader(
            TensorDataset(train_t),
            batch_size=batch_size,
            shuffle=True,
            num_workers=0,
            pin_memory=(device != "cpu"),
        )

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)

    history = {"train_loss": [], "val_loss": [], "val_acc": [], "val_bpb": []}
    best_val_loss = float("inf")

    for epoch in range(n_epochs):
        t0 = time.time()
        model.train()
        total_loss = 0
        n_batches = 0
        for (batch,) in train_dl:
            batch = batch.to(device)
            logits, z = model(batch)
            loss = F.cross_entropy(
                logits.reshape(-1, NUM_CLASSES),
                batch.reshape(-1),
                ignore_index=0,
            )
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            n_batches += 1

        scheduler.step()
        avg_train = total_loss / n_batches

        # Validation (in chunks to avoid OOM)
        model.eval()
        with torch.no_grad():
            val_ce_sum = 0.0
            val_correct = 0
            val_log2p_sum = 0.0
            val_nonpad = 0
            chunk = 8192
            for i in range(0, len(val_t), chunk):
                vb = val_t[i : i + chunk]
                vl, vz = model(vb)
                non_pad = vb > 0
                n_np = non_pad.sum().item()
                val_nonpad += n_np

                ce = F.cross_entropy(
                    vl.reshape(-1, NUM_CLASSES),
                    vb.reshape(-1),
                    ignore_index=0,
                    reduction="sum",
                ).item()
                val_ce_sum += ce
                val_correct += (vl.argmax(-1)[non_pad] == vb[non_pad]).sum().item()

                lp = F.log_softmax(vl, dim=-1)
                blp = lp.gather(2, vb.unsqueeze(-1)).squeeze(-1)
                val_log2p_sum += blp[non_pad].sum().item()

        val_loss = val_ce_sum / val_nonpad
        val_acc = val_correct / val_nonpad
        val_bpb = -val_log2p_sum / val_nonpad / np.log(2)
        dt = time.time() - t0

        history["train_loss"].append(avg_train)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        history["val_bpb"].append(val_bpb)

        marker = " *" if val_loss < best_val_loss else ""
        best_val_loss = min(best_val_loss, val_loss)
        print(
            f"  [{label}] Epoch {epoch + 1:2d}/{n_epochs} ({dt:.1f}s): "
            f"train={avg_train:.4f}  val={val_loss:.4f}  "
            f"acc={val_acc:.3f}  bpb={val_bpb:.4f}{marker}"
        )

    return history


def show_examples(model, data, n=20):
    """Show reconstruction examples."""
    model.eval()
    sample = torch.from_numpy(data[:n].astype(np.int64)).to(device)
    with torch.no_grad():
        logits, z = model(sample)
        preds = logits.argmax(dim=-1)

    print(f"\n--- Reconstructions ---")
    n_correct = 0
    for i in range(len(sample)):
        orig = sample[i].cpu().numpy()
        pred = preds[i].cpu().numpy()
        orig_len = (orig > 0).sum()
        orig_w = bytes(orig[:orig_len]).decode("utf-8", errors="replace")
        pred_w = bytes(pred[:orig_len]).decode("utf-8", errors="replace")
        ok = orig_w == pred_w
        n_correct += ok
        tag = "OK" if ok else "XX"
        print(f"  {tag} {orig_w:>25s} -> {pred_w:<25s} |z|={z[i].norm():.2f}")
    print(f"  {n_correct}/{len(sample)} correct")

### Experiment 1: MLP across latent dimensions

In [ ]:
LATENT_DIMS = [32]
N_EPOCHS = 1
HIDDEN_DIM = 32
EMBED_DIM = 32
BATCH_SIZE = 128

mlp_results = {}

for ld in LATENT_DIMS:
    print(f"\n{'=' * 60}")
    print(f"MLP latent_dim={ld}")
    print(f"{'=' * 60}")
    model = MLPAutoencoder(
        max_len=MAX_WORD_LEN,
        embed_dim=EMBED_DIM,
        hidden_dim=HIDDEN_DIM,
        latent_dim=ld,
    )
    print(f"Parameters: {count_params(model):,}")
    hist = train_autoencoder(
        model,
        train_words,
        val_words,
        n_epochs=N_EPOCHS,
        batch_size=BATCH_SIZE,
        label=f"MLP-{ld}",
        train_weights=train_word_weights,
    )
    mlp_results[ld] = {"history": hist, "model": model}

# Show examples for best latent dim
best_ld = min(mlp_results, key=lambda d: mlp_results[d]["history"]["val_bpb"][-1])
print(f"\nBest MLP latent_dim={best_ld}")
show_examples(mlp_results[best_ld]["model"], val_words)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Training curves
ax = axes[0]
for ld, res in sorted(mlp_results.items()):
    ax.plot(res["history"]["val_loss"], label=f"d={ld}")
ax.set_xlabel("Epoch")
ax.set_ylabel("Val loss (CE)")
ax.set_title("MLP: validation loss by latent dim")
ax.legend()
ax.grid(True, alpha=0.3)

# Final BPB vs latent dim
ax = axes[1]
lds = sorted(mlp_results.keys())
final_bpb = [mlp_results[d]["history"]["val_bpb"][-1] for d in lds]
ax.plot(lds, final_bpb, "go-", lw=2, markersize=8)
for d, b in zip(lds, final_bpb):
    ax.annotate(
        f"{b:.3f}",
        (d, b),
        textcoords="offset points",
        xytext=(0, 10),
        ha="center",
        fontsize=9,
    )
ax.set_xlabel("Latent dimension")
ax.set_ylabel("Val BPB")
ax.set_title("MLP: final BPB vs latent dim")
ax.grid(True, alpha=0.3)

# Final accuracy vs latent dim
ax = axes[2]
final_acc = [mlp_results[d]["history"]["val_acc"][-1] for d in lds]
ax.plot(lds, final_acc, "ms-", lw=2, markersize=8)
for d, a in zip(lds, final_acc):
    ax.annotate(
        f"{a:.3f}",
        (d, a),
        textcoords="offset points",
        xytext=(0, 10),
        ha="center",
        fontsize=9,
    )
ax.set_xlabel("Latent dimension")
ax.set_ylabel("Val accuracy")
ax.set_title("MLP: byte accuracy vs latent dim")
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(PLOT_DIR / "word_ae_mlp_latent_sweep.png", dpi=150, bbox_inches="tight")
plt.show()

### Experiment 1b: Conv across latent dimensions

In [ ]:
LATENT_DIMS_CONV = [128]
N_EPOCHS_CONV = 15
CHANNELS = 128
EMBED_DIM_CONV = 128
N_CONVS = 8
N_DOWNSAMPLE = 4

# Sweep latent dim with default depth (3 conv, 2 downsample)
conv_results = {}

for ld in LATENT_DIMS_CONV:
    print(f"\n{'=' * 60}")
    print(f"Conv latent_dim={ld}")
    print(f"{'=' * 60}")
    model = ConvAutoencoder(
        max_len=MAX_WORD_LEN,
        embed_dim=EMBED_DIM_CONV,
        channels=CHANNELS,
        latent_dim=ld,
        n_conv=N_CONVS,
        n_downsample=N_DOWNSAMPLE,
    )
    print(f"Parameters: {count_params(model):,}  |  {model}")
    hist = train_autoencoder(
        model,
        train_words,
        val_words,
        n_epochs=N_EPOCHS_CONV,
        batch_size=BATCH_SIZE,
        label=f"Conv-{ld}",
        train_weights=train_word_weights,
    )
    conv_results[ld] = {"history": hist, "model": model}

# Sweep depth at fixed latent dim
LATENT_DIM_DEPTH = 32
depth_configs = [
    ("2c-1d", {"n_conv": 2, "n_downsample": 1}),
    # ("3c-2d", {"n_conv": 3, "n_downsample": 2}),
    # ("3c-2d-tied", {"n_conv": 3, "n_downsample": 2, "tied_weights": True}),
    # ("4c-2d", {"n_conv": 4, "n_downsample": 2}),
    # ("4c-2d-tied", {"n_conv": 4, "n_downsample": 2, "tied_weights": True}),
    # ("4c-3d", {"n_conv": 4, "n_downsample": 3}),
    # ("4c-3d-tied", {"n_conv": 4, "n_downsample": 3, "tied_weights": True}),
    # ("5c-3d", {"n_conv": 5, "n_downsample": 3}),
    # ("5c-3d-tied", {"n_conv": 5, "n_downsample": 3, "tied_weights": True}),
]

conv_depth_results = {}

for name, kwargs in depth_configs:
    print(f"\n{'=' * 60}")
    print(f"Conv {name} (latent_dim={LATENT_DIM_DEPTH})")
    print(f"{'=' * 60}")
    model = ConvAutoencoder(
        max_len=MAX_WORD_LEN,
        embed_dim=EMBED_DIM_CONV,
        channels=CHANNELS,
        latent_dim=LATENT_DIM_DEPTH,
        **kwargs,
    )
    print(f"Parameters: {count_params(model):,}  |  {model}")
    hist = train_autoencoder(
        model,
        train_words,
        val_words,
        n_epochs=N_EPOCHS_CONV,
        batch_size=BATCH_SIZE,
        label=name,
        train_weights=train_word_weights,
    )
    conv_depth_results[name] = {
        "history": hist,
        "model": model,
        "params": count_params(model),
        **kwargs,
    }

# Show examples for best
best_ld = min(conv_results, key=lambda d: conv_results[d]["history"]["val_bpb"][-1])
print(f"\nBest Conv latent_dim={best_ld}")
show_examples(conv_results[best_ld]["model"], val_words)

best_depth = min(
    conv_depth_results, key=lambda n: conv_depth_results[n]["history"]["val_bpb"][-1]
)
print(f"\nBest Conv depth: {best_depth}")
show_examples(conv_depth_results[best_depth]["model"], val_words)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# --- Top row: latent dim sweep ---
ax = axes[0, 0]
for ld, res in sorted(conv_results.items()):
    ax.plot(res["history"]["val_loss"], label=f"d={ld}")
ax.set_xlabel("Epoch")
ax.set_ylabel("Val loss (CE)")
ax.set_title("Conv: val loss by latent dim")
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[0, 1]
lds = sorted(conv_results.keys())
final_bpb = [conv_results[d]["history"]["val_bpb"][-1] for d in lds]
ax.plot(lds, final_bpb, "ro-", lw=2, markersize=8)
for d, b in zip(lds, final_bpb):
    ax.annotate(
        f"{b:.3f}",
        (d, b),
        textcoords="offset points",
        xytext=(0, 10),
        ha="center",
        fontsize=9,
    )
ax.set_xlabel("Latent dimension")
ax.set_ylabel("Val BPB")
ax.set_title("Conv: BPB vs latent dim")
ax.grid(True, alpha=0.3)

ax = axes[0, 2]
final_acc = [conv_results[d]["history"]["val_acc"][-1] for d in lds]
ax.plot(lds, final_acc, "ms-", lw=2, markersize=8)
for d, a in zip(lds, final_acc):
    ax.annotate(
        f"{a:.3f}",
        (d, a),
        textcoords="offset points",
        xytext=(0, 10),
        ha="center",
        fontsize=9,
    )
ax.set_xlabel("Latent dimension")
ax.set_ylabel("Val accuracy")
ax.set_title("Conv: accuracy vs latent dim")
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)

# --- Bottom row: depth sweep ---
ax = axes[1, 0]
for name, res in conv_depth_results.items():
    ax.plot(res["history"]["val_loss"], label=name)
ax.set_xlabel("Epoch")
ax.set_ylabel("Val loss (CE)")
ax.set_title(f"Conv depth: val loss (d={LATENT_DIM_DEPTH})")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

ax = axes[1, 1]
names = list(conv_depth_results.keys())
bpbs = [conv_depth_results[n]["history"]["val_bpb"][-1] for n in names]
params = [conv_depth_results[n]["params"] for n in names]
ax.bar(range(len(names)), bpbs, color="coral", alpha=0.7)
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, rotation=45, ha="right", fontsize=8)
for i, (b, p) in enumerate(zip(bpbs, params)):
    ax.text(i, b + 0.01, f"{b:.3f}\n({p // 1000}k)", ha="center", fontsize=8)
ax.set_ylabel("Val BPB")
ax.set_title("Conv depth: final BPB")
ax.grid(True, alpha=0.3)

ax = axes[1, 2]
accs = [conv_depth_results[n]["history"]["val_acc"][-1] for n in names]
ax.bar(range(len(names)), accs, color="steelblue", alpha=0.7)
ax.set_xticks(range(len(names)))
ax.set_xticklabels(names, rotation=45, ha="right", fontsize=8)
for i, a in enumerate(accs):
    ax.text(i, a + 0.01, f"{a:.3f}", ha="center", fontsize=8)
ax.set_ylabel("Val accuracy")
ax.set_title("Conv depth: byte accuracy")
ax.set_ylim(0, 1)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(PLOT_DIR / "word_ae_conv_latent_sweep.png", dpi=150, bbox_inches="tight")
plt.show()

# Comparison table
if mlp_results:
    print(
        f"\n{'Latent':>8s} {'MLP BPB':>10s} {'Conv BPB':>10s} {'MLP Acc':>10s} {'Conv Acc':>10s}"
    )
    print("-" * 50)
    for ld in sorted(set(mlp_results.keys()) & set(conv_results.keys())):
        mb = mlp_results[ld]["history"]["val_bpb"][-1]
        cb = conv_results[ld]["history"]["val_bpb"][-1]
        ma = mlp_results[ld]["history"]["val_acc"][-1]
        ca = conv_results[ld]["history"]["val_acc"][-1]
        print(f"{ld:8d} {mb:10.4f} {cb:10.4f} {ma:10.3f} {ca:10.3f}")

### Experiment 2: Architecture comparison (MLP vs Conv vs GRU)

In [ ]:
LATENT_DIM_CMP = 32
N_EPOCHS_CMP = 10

arch_configs = {
    "MLP": MLPAutoencoder(
        max_len=MAX_WORD_LEN, embed_dim=32, hidden_dim=64, latent_dim=LATENT_DIM_CMP
    ),
    "Conv": ConvAutoencoder(
        max_len=MAX_WORD_LEN, embed_dim=32, channels=64, latent_dim=LATENT_DIM_CMP
    ),
    "GRU": GRUAutoencoder(
        max_len=MAX_WORD_LEN, embed_dim=32, hidden_dim=64, latent_dim=LATENT_DIM_CMP
    ),
}

arch_results = {}

for name, model in arch_configs.items():
    print(f"\n{'=' * 60}")
    print(f"{name} (latent_dim={LATENT_DIM_CMP}, params={count_params(model):,})")
    print(f"{'=' * 60}")
    hist = train_autoencoder(
        model,
        train_words,
        val_words,
        n_epochs=N_EPOCHS_CMP,
        batch_size=BATCH_SIZE,
        label=name,
        train_weights=train_word_weights,
    )
    arch_results[name] = {
        "history": hist,
        "model": model,
        "params": count_params(model),
    }
    show_examples(model, val_words, n=10)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = {"MLP": "#2080d0", "Conv": "#d04020", "GRU": "#20a040"}

# Val loss curves
ax = axes[0]
for name, res in arch_results.items():
    ax.plot(res["history"]["val_loss"], label=name, color=colors[name], lw=2)
ax.set_xlabel("Epoch")
ax.set_ylabel("Val loss (CE)")
ax.set_title(f"Architecture comparison (d={LATENT_DIM_CMP})")
ax.legend()
ax.grid(True, alpha=0.3)

# Val BPB curves
ax = axes[1]
for name, res in arch_results.items():
    ax.plot(res["history"]["val_bpb"], label=name, color=colors[name], lw=2)
ax.set_xlabel("Epoch")
ax.set_ylabel("Val BPB")
ax.set_title("BPB over training")
ax.legend()
ax.grid(True, alpha=0.3)

# Final BPB vs params
ax = axes[2]
for name, res in arch_results.items():
    ax.scatter(
        res["params"],
        res["history"]["val_bpb"][-1],
        s=150,
        c=colors[name],
        label=name,
        zorder=5,
    )
    ax.annotate(
        f"{res['history']['val_bpb'][-1]:.3f}",
        (res["params"], res["history"]["val_bpb"][-1]),
        textcoords="offset points",
        xytext=(10, 5),
        fontsize=10,
    )
ax.set_xlabel("Parameters")
ax.set_ylabel("Final val BPB")
ax.set_title("Efficiency: BPB vs model size")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(PLOT_DIR / "word_ae_arch_compare.png", dpi=150, bbox_inches="tight")
plt.show()

# Summary table
print(f"\n{'Architecture':<12s} {'Params':>10s} {'Val BPB':>10s} {'Val Acc':>10s}")
print("-" * 45)
for name, res in arch_results.items():
    h = res["history"]
    print(
        f"{name:<12s} {res['params']:>10,} {h['val_bpb'][-1]:>10.4f} "
        f"{h['val_acc'][-1]:>10.3f}"
    )

### Latent space visualization

In [ ]:
# ================================================================
# Visualize and compare latent spaces across architectures
# ================================================================
from sklearn.decomposition import PCA

# Encode a large sample of val words through each model
N_SAMPLE = min(10_000, len(val_words))
sample_data = torch.from_numpy(val_words[:N_SAMPLE].astype(np.int64)).to(device)
sample_lengths = (val_words[:N_SAMPLE] > 0).sum(axis=1)

# Decode words for coloring
sample_words_bytes = []
for i in range(N_SAMPLE):
    wl = sample_lengths[i]
    sample_words_bytes.append(bytes(val_words[i, :wl]))

# Collect latent vectors from all trained architectures
latent_vecs = {}
for name, res in arch_results.items():
    model = res["model"].to(device)
    model.eval()
    with torch.no_grad():
        _, z = model(sample_data)
        latent_vecs[name] = z.cpu().numpy()

n_arch = len(latent_vecs)

# --- Figure 1: PCA projections colored by word length ---
fig, axes = plt.subplots(1, n_arch, figsize=(6 * n_arch, 5))
if n_arch == 1:
    axes = [axes]

for ax, (name, z) in zip(axes, latent_vecs.items()):
    pca = PCA(n_components=2)
    z2d = pca.fit_transform(z)
    sc = ax.scatter(
        z2d[:, 0],
        z2d[:, 1],
        c=sample_lengths[:N_SAMPLE],
        cmap="viridis",
        s=3,
        alpha=0.3,
    )
    ax.set_title(f"{name} — colored by word length")
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")
    plt.colorbar(sc, ax=ax, label="Word length")

plt.tight_layout()
plt.savefig(PLOT_DIR / "word_ae_latent_pca_length.png", dpi=150, bbox_inches="tight")
plt.show()

# --- Figure 2: PCA projections colored by first letter ---
first_letters = np.array([w[0] if len(w) > 0 else 0 for w in sample_words_bytes])
# Map to 0-25 for a-z (lowercase)
first_letter_idx = np.array(
    [
        (b - ord("a"))
        if ord("a") <= b <= ord("z")
        else (b - ord("A"))
        if ord("A") <= b <= ord("Z")
        else 26
        for b in first_letters
    ]
)

fig, axes = plt.subplots(1, n_arch, figsize=(6 * n_arch, 5))
if n_arch == 1:
    axes = [axes]

for ax, (name, z) in zip(axes, latent_vecs.items()):
    pca = PCA(n_components=2)
    z2d = pca.fit_transform(z)
    # Only plot letters a-z for cleaner visualization
    mask = first_letter_idx < 26
    sc = ax.scatter(
        z2d[mask, 0],
        z2d[mask, 1],
        c=first_letter_idx[mask],
        cmap="tab20",
        s=3,
        alpha=0.3,
        vmin=0,
        vmax=25,
    )
    ax.set_title(f"{name} — colored by first letter")
    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%})")
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%})")

plt.tight_layout()
plt.savefig(PLOT_DIR / "word_ae_latent_pca_letter.png", dpi=150, bbox_inches="tight")
plt.show()

# --- Figure 3: Per-dimension statistics ---
fig, axes = plt.subplots(2, n_arch, figsize=(6 * n_arch, 8))
if n_arch == 1:
    axes = axes.reshape(-1, 1)

for col, (name, z) in enumerate(latent_vecs.items()):
    d = z.shape[1]

    # Top: mean and std per dimension
    ax = axes[0, col]
    means = z.mean(axis=0)
    stds = z.std(axis=0)
    x = np.arange(d)
    ax.bar(x, means, yerr=stds, capsize=2, alpha=0.7, color="steelblue")
    ax.set_xlabel("Latent dimension")
    ax.set_ylabel("Mean ± std")
    ax.set_title(f"{name}: per-dim statistics (d={d})")
    ax.grid(True, alpha=0.3)

    # Bottom: correlation matrix
    ax = axes[1, col]
    corr = np.corrcoef(z.T)
    im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
    ax.set_title(f"{name}: dim correlation")
    ax.set_xlabel("Dimension")
    ax.set_ylabel("Dimension")
    plt.colorbar(im, ax=ax)

plt.tight_layout()
plt.savefig(PLOT_DIR / "word_ae_latent_stats.png", dpi=150, bbox_inches="tight")
plt.show()

# --- Figure 4: Variance explained by PCA ---
fig, ax = plt.subplots(figsize=(8, 5))
for name, z in latent_vecs.items():
    d = z.shape[1]
    pca_full = PCA(n_components=d)
    pca_full.fit(z)
    cumvar = np.cumsum(pca_full.explained_variance_ratio_)
    ax.plot(np.arange(1, d + 1), cumvar, "o-", label=name, markersize=4)

ax.set_xlabel("Number of principal components")
ax.set_ylabel("Cumulative variance explained")
ax.set_title("Latent space effective dimensionality")
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig(PLOT_DIR / "word_ae_latent_variance.png", dpi=150, bbox_inches="tight")
plt.show()

# --- Summary ---
print(
    f"\n{'Architecture':<12s} {'Latent d':>10s} {'Active dims':>12s} "
    f"{'|z| mean':>10s} {'|z| std':>10s}"
)
print("-" * 58)
for name, z in latent_vecs.items():
    d = z.shape[1]
    pca_full = PCA(n_components=d).fit(z)
    # "Active dims" = number of PCs needed for 95% variance
    cumvar = np.cumsum(pca_full.explained_variance_ratio_)
    active = int(np.searchsorted(cumvar, 0.95)) + 1
    norms = np.linalg.norm(z, axis=1)
    print(f"{name:<12s} {d:10d} {active:12d} {norms.mean():10.2f} {norms.std():10.2f}")

In [ ]:
# ================================================================
# Compression analysis: is the autoencoder actually compressing?
# ================================================================
# Compare the information content in the latent vector vs the input.
# If the encoder is just memorizing/passing through the embedding,
# the latent will have high mutual information with the input embedding
# and low actual compression.

# Metrics:
# 1. Compression ratio: input bytes vs latent floats
# 2. Reconstruction from truncated latent (zero out dims by variance)
# 3. Linear probe: can we recover the input embedding from z?
# 4. Nearest-neighbor: do similar words have similar latents?

print("=" * 60)
print("Compression analysis")
print("=" * 60)

N_SAMPLE = min(10_000, len(val_words))
sample_data = torch.from_numpy(val_words[:N_SAMPLE].astype(np.int64)).to(device)
sample_lengths = (val_words[:N_SAMPLE] > 0).sum(axis=1)

# --- 1. Compression ratio ---
print(f"\n--- Compression ratios ---")
print(
    f"{'Model':<12s} {'Input':>12s} {'Latent':>12s} {'Ratio':>8s} "
    f"{'Eff. dims':>10s} {'Eff. ratio':>10s}"
)
print("-" * 68)

for name, res in arch_results.items():
    model = res["model"].to(device)
    model.eval()
    d = None
    for p_name, p in model.named_parameters():
        if "enc_fc.weight" in p_name or "encoder" in p_name:
            pass
    # Get latent dim
    with torch.no_grad():
        _, z = model(sample_data[:1])
        d = z.shape[1]

    # Input: mean word length in bytes
    mean_len = sample_lengths.mean()
    # Latent: d float32 values
    input_bits = mean_len * 8  # 8 bits per byte
    latent_bits_f32 = d * 32
    latent_bits_f16 = d * 16

    # Effective dims from PCA (95% variance)
    with torch.no_grad():
        _, z_all = model(sample_data)
        z_np = z_all.cpu().numpy()
    from sklearn.decomposition import PCA

    pca = PCA(n_components=d).fit(z_np)
    cumvar = np.cumsum(pca.explained_variance_ratio_)
    eff_dims_95 = int(np.searchsorted(cumvar, 0.95)) + 1
    eff_bits = eff_dims_95 * 16  # assume fp16 for effective dims

    ratio_f32 = input_bits / latent_bits_f32
    eff_ratio = input_bits / eff_bits

    print(
        f"{name:<12s} {mean_len:>9.1f} B   {d:>8d} f32 {ratio_f32:>8.2f} "
        f"{eff_dims_95:>10d} {eff_ratio:>10.2f}"
    )

# --- 2. Reconstruction from truncated latent ---
# Zero out latent dimensions ordered by variance (least important first)
# and measure how BPB degrades
print(f"\n--- Reconstruction vs active latent dimensions ---")

fig, axes = plt.subplots(1, len(arch_results), figsize=(6 * len(arch_results), 5))
if len(arch_results) == 1:
    axes = [axes]

for ax, (name, res) in zip(axes, arch_results.items()):
    model = res["model"].to(device)
    model.eval()

    with torch.no_grad():
        _, z_all = model(sample_data)
        z_np = z_all.cpu().numpy()

    d = z_np.shape[1]
    pca = PCA(n_components=d).fit(z_np)

    # Project to PCA space, then reconstruct with increasing number of components
    z_pca = pca.transform(z_np)
    dims_to_test = sorted(
        set([1, 2, 4, 8, 12, 16, 24, 32, 48, 64, 96, 128]) & set(range(1, d + 1))
    )
    dims_to_test.append(d)
    dims_to_test = sorted(set(dims_to_test))

    bpbs = []
    accs = []
    for n_dims in dims_to_test:
        # Keep top n_dims PCA components, zero rest
        z_trunc = z_pca.copy()
        z_trunc[:, n_dims:] = 0
        z_recon = pca.inverse_transform(z_trunc)

        z_t = torch.from_numpy(z_recon.astype(np.float32)).to(device)
        with torch.no_grad():
            logits = model.decode(z_t)
            non_pad = sample_data > 0
            n_np = non_pad.sum().item()

            lp = F.log_softmax(logits, dim=-1)
            blp = lp.gather(2, sample_data.unsqueeze(-1)).squeeze(-1)
            bpb = -(blp[non_pad].sum().item() / n_np) / np.log(2)

            preds = logits.argmax(dim=-1)
            acc = (preds[non_pad] == sample_data[non_pad]).float().mean().item()

        bpbs.append(bpb)
        accs.append(acc)

    ax.plot(dims_to_test, bpbs, "o-", color="steelblue", lw=2, label="BPB")
    ax2 = ax.twinx()
    ax2.plot(dims_to_test, accs, "s--", color="coral", lw=2, label="Accuracy")
    ax2.set_ylabel("Byte accuracy", color="coral")
    ax2.set_ylim(0, 1)
    ax.set_xlabel("Active PCA dimensions")
    ax.set_ylabel("BPB", color="steelblue")
    ax.set_title(f"{name} (d={d})")
    ax.grid(True, alpha=0.3)

    # Mark the "knee" — where we get within 5% of full BPB
    full_bpb = bpbs[-1]
    for i, (nd, b) in enumerate(zip(dims_to_test, bpbs)):
        if b < full_bpb * 1.05:
            ax.axvline(nd, color="green", ls=":", alpha=0.5)
            ax.annotate(
                f"~{nd}d",
                (nd, b),
                textcoords="offset points",
                xytext=(5, 10),
                color="green",
                fontsize=10,
            )
            break

plt.tight_layout()
plt.savefig(PLOT_DIR / "word_ae_latent_compression.png", dpi=150, bbox_inches="tight")
plt.show()

# --- 3. Linear probe: can z recover the flat input embedding? ---
# If R² is very high, the encoder is mostly doing a linear projection
# (not learning a compressed representation)
print(f"\n--- Linear probe: z -> input embedding ---")
from sklearn.linear_model import Ridge

for name, res in arch_results.items():
    model = res["model"].to(device)
    model.eval()
    with torch.no_grad():
        emb = model.embed(sample_data)  # (N, max_len, embed_dim)
        emb_flat = emb.reshape(N_SAMPLE, -1).cpu().numpy()
        _, z = model(sample_data)
        z_np = z.cpu().numpy()

    # Fit linear map z -> emb_flat on 80% train, evaluate on 20%
    n_tr = int(0.8 * N_SAMPLE)
    reg = Ridge(alpha=1.0).fit(z_np[:n_tr], emb_flat[:n_tr])
    pred_emb = reg.predict(z_np[n_tr:])
    true_emb = emb_flat[n_tr:]

    # R² score
    ss_res = ((true_emb - pred_emb) ** 2).sum()
    ss_tot = ((true_emb - true_emb.mean(axis=0)) ** 2).sum()
    r2 = 1 - ss_res / ss_tot

    # Also measure per-position R²
    emb_dim = emb.shape[-1]
    pos_r2 = []
    for pos in range(MAX_WORD_LEN):
        true_pos = true_emb[:, pos * emb_dim : (pos + 1) * emb_dim]
        pred_pos = pred_emb[:, pos * emb_dim : (pos + 1) * emb_dim]
        ss_r = ((true_pos - pred_pos) ** 2).sum()
        ss_t = ((true_pos - true_pos.mean(axis=0)) ** 2).sum()
        pos_r2.append(1 - ss_r / ss_t if ss_t > 0 else 0)

    compression = (MAX_WORD_LEN * emb_dim) / z_np.shape[1]
    print(
        f"  {name}: R²={r2:.4f}  compression={compression:.1f}x  "
        f"(z:{z_np.shape[1]}d -> emb:{MAX_WORD_LEN}x{emb_dim}={MAX_WORD_LEN * emb_dim}d)"
    )
    if r2 > 0.9:
        print(
            f"    ⚠ High R² suggests encoder is mostly a linear projection, "
            f"not learning deep compression"
        )
    elif r2 < 0.5:
        print(f"    ✓ Low R² — encoder learns a non-trivial compressed representation")

# --- 4. Nearest-neighbor sanity check ---
print(f"\n--- Nearest neighbors in latent space ---")
from sklearn.neighbors import NearestNeighbors

for name, res in arch_results.items():
    model = res["model"].to(device)
    model.eval()
    with torch.no_grad():
        _, z = model(sample_data)
        z_np = z.cpu().numpy()

    nn_model = NearestNeighbors(n_neighbors=6, metric="euclidean")
    nn_model.fit(z_np)

    # Pick 10 random query words
    rng = np.random.default_rng(42)
    query_idx = rng.choice(N_SAMPLE, 10, replace=False)
    dists, indices = nn_model.kneighbors(z_np[query_idx])

    print(f"\n  {name} — nearest neighbors:")
    for qi, (ds, idxs) in enumerate(zip(dists, indices)):
        wl = sample_lengths[query_idx[qi]]
        query_w = bytes(val_words[query_idx[qi], :wl]).decode("utf-8", errors="replace")
        neighbors = []
        for j, (d, ni) in enumerate(zip(ds[1:], idxs[1:])):  # skip self
            nl = sample_lengths[ni]
            nw = bytes(val_words[ni, :nl]).decode("utf-8", errors="replace")
            neighbors.append(f"{nw} ({d:.2f})")
        print(f"    {query_w:>20s} → {', '.join(neighbors)}")

## Analysis: reconstruction quality by sequence length

In [ ]:
# Analyze best model's reconstruction quality by word length
best_name = min(arch_results, key=lambda n: arch_results[n]["history"]["val_bpb"][-1])
best_model = arch_results[best_name]["model"].to(device)
best_model.eval()

print(f"Analyzing {best_name} (best val BPB)...")

val_t = torch.from_numpy(val_words.astype(np.int64)).to(device)
val_lengths = (val_words > 0).sum(axis=1)

# Compute per-word metrics
word_bpbs = []
word_accs = []
word_exact = []

chunk = 4096
with torch.no_grad():
    for i in range(0, len(val_t), chunk):
        vb = val_t[i : i + chunk]
        vl, vz = best_model(vb)
        preds = vl.argmax(dim=-1)
        lp = F.log_softmax(vl, dim=-1)
        blp = lp.gather(2, vb.unsqueeze(-1)).squeeze(-1)

        for j in range(len(vb)):
            wl = int(val_lengths[i + j])
            if wl == 0:
                continue
            bpb_j = -blp[j, :wl].sum().item() / wl / np.log(2)
            acc_j = (preds[j, :wl] == vb[j, :wl]).float().mean().item()
            exact = (preds[j, :wl] == vb[j, :wl]).all().item()
            word_bpbs.append((wl, bpb_j))
            word_accs.append((wl, acc_j))
            word_exact.append((wl, exact))

# Aggregate by length
from collections import defaultdict

bpb_by_len = defaultdict(list)
acc_by_len = defaultdict(list)
exact_by_len = defaultdict(list)
for l, b in word_bpbs:
    bpb_by_len[l].append(b)
for l, a in word_accs:
    acc_by_len[l].append(a)
for l, e in word_exact:
    exact_by_len[l].append(e)

lengths_sorted = sorted(bpb_by_len.keys())
mean_bpb = [np.mean(bpb_by_len[l]) for l in lengths_sorted]
mean_acc = [np.mean(acc_by_len[l]) for l in lengths_sorted]
mean_exact = [np.mean(exact_by_len[l]) for l in lengths_sorted]
counts = [len(bpb_by_len[l]) for l in lengths_sorted]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

ax = axes[0]
ax.plot(lengths_sorted, mean_bpb, "go-", lw=2)
ax.set_xlabel(f"{MODE.title()} length")
ax.set_ylabel("Mean BPB")
ax.set_title(f"{best_name}: BPB by word length")
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(lengths_sorted, mean_acc, "bs-", lw=2, label="Byte accuracy")
ax.plot(lengths_sorted, mean_exact, "r^--", lw=2, label="Exact match")
ax.set_xlabel(f"{MODE.title()} length")
ax.set_ylabel("Rate")
ax.set_title(f"{best_name}: accuracy by word length")
ax.set_ylim(0, 1)
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[2]
ax.bar(lengths_sorted, counts, color="gray", alpha=0.7)
ax.set_xlabel(f"{MODE.title()} length")
ax.set_ylabel("Count")
ax.set_title(f"Val {MODE} length distribution")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(PLOT_DIR / f"byte_ae_{MODE}_by_length.png", dpi=150, bbox_inches="tight")
plt.show()

# Print table
print(f"\n{'Len':>4s} {'Count':>8s} {'BPB':>8s} {'ByteAcc':>8s} {'Exact':>8s}")
print("-" * 40)
for l, b, a, e, c in zip(lengths_sorted, mean_bpb, mean_acc, mean_exact, counts):
    print(f"{l:4d} {c:8,} {b:8.4f} {a:8.3f} {e:8.3f}")